In [ ]:
import numpy as np

def nu2(x): return (x & -x).bit_length() - 1
def sgn(i, j):
    if i == 0 or j == 0: return 1
    if i == j: return -1
    t, ti, tj = nu2(i ^ j), nu2(i), nu2(j)
    if t >= ti and t >= tj: k = t + 1; s = 2*((i >> t) & 1) - 1
    elif ti >= tj:          k = ti + 1; s = 2*((j >> ti) & 1) - 1
    else:                   k = tj + 1; s = 1 - 2*((i >> tj) & 1)
    if ((i | j) >> k).bit_count() & 1: s = -s
    return s

def label_level(M):
    h, w = M.shape
    alpha, out = {}, np.zeros((h//2, w//2), dtype=np.int32)
    for a in range(h//2):
        for b in range(w//2):
            key = tuple(int(v) for v in M[2*a:2*a+2, 2*b:2*b+2].ravel())
            if key not in alpha: alpha[key] = len(alpha) + 1
            out[a, b] = alpha[key]
    return out, alpha

n = 10
d = 1 << n
M = np.array([[sgn(i, j) for j in range(d)] for i in range(d)], dtype=np.int8)
for lvl in range(1, n + 1):
    M, alpha = label_level(M)
    print(f"level {lvl}: alphabet = {len(alpha):4d}   matrix {M.shape}")

level 1: alphabet =    6   matrix (512, 512)
level 2: alphabet =   12   matrix (256, 256)
level 3: alphabet =   12   matrix (128, 128)
level 4: alphabet =   12   matrix (64, 64)
level 5: alphabet =   12   matrix (32, 32)
level 6: alphabet =   12   matrix (16, 16)
level 7: alphabet =   12   matrix (8, 8)
level 8: alphabet =    9   matrix (4, 4)
level 9: alphabet =    4   matrix (2, 2)
level 10: alphabet =    1   matrix (1, 1)


In [ ]:
import numpy as np, math
from collections import defaultdict

def nu2(x): return (x & -x).bit_length() - 1
BIG = 99

def feats(a, b, mode):
    za = nu2(a) if a else BIG
    zb = nu2(b) if b else BIG
    zx = nu2(a ^ b) if (a ^ b) else BIG
    ab = a | b
    base = (a == 0, b == 0, a == b, min(za,3), min(zb,3), min(zx,3),
            za < zb, zb < zx, zx < za,
            bin(a).count("1") & 1, bin(b).count("1") & 1)
    if mode >= 1:                      # + unshifted overlap parities
        base += (bin(ab).count("1") & 1, bin(a & b).count("1") & 1)
    if mode >= 2:                      # + SHIFTED OR parities (the real terms)
        pa = bin(ab >> za).count("1") & 1 if za != BIG else 0
        pb = bin(ab >> zb).count("1") & 1 if zb != BIG else 0
        px = bin(ab >> zx).count("1") & 1 if zx != BIG else 0
        base += (pa, pb, px)
    if mode >= 3:                      # + full order type with ties
        base += (za == zb, zb == zx, za == zx, za < zx, zb < za, zx < zb)
    return base

L = mats[3]                            # 64x64 plateau matrix, 4096 cells
h = L.shape[0]; total = h * h
for mode in (0, 1, 2, 3):
    buckets = defaultdict(list)
    for a in range(h):
        for b in range(h):
            buckets[feats(a, b, mode)].append(int(L[a, b]))
    impure_cells = sum(len(v) for v in buckets.values() if len(set(v)) > 1)
    # H(symbol | features), in bits
    H = 0.0
    for v in buckets.values():
        cnt = defaultdict(int)
        for s in v: cnt[s] += 1
        n = len(v)
        H += (n / total) * (-sum((c/n) * math.log2(c/n) for c in cnt.values()))
    print(f"mode={mode}: classes={len(buckets):4d} "
          f"impure_cells={impure_cells:5d}/{total} "
          f"H(symbol|feat)={H:.4f} bits  determined={impure_cells==0}")

mode=0: classes= 109 impure_cells= 3906/4096 H(symbol|feat)=1.8819 bits  determined=False
mode=1: classes= 187 impure_cells= 3888/4096 H(symbol|feat)=0.9368 bits  determined=False
mode=2: classes= 283 impure_cells= 3690/4096 H(symbol|feat)=0.8523 bits  determined=False
mode=3: classes= 283 impure_cells= 3690/4096 H(symbol|feat)=0.8523 bits  determined=False


In [ ]:
from collections import defaultdict

# --- Test F: consistency across plateau levels and dimensions ---
def feats(a, b):
    BIG = 99
    za = nu2(a) if a else BIG
    zb = nu2(b) if b else BIG
    zx = nu2(a ^ b) if (a ^ b) else BIG
    ab = a | b
    return (a == 0, b == 0, a == b, min(za,3), min(zb,3), min(zx,3),
            za < zb, zb < zx, zx < za,
            bin(a).count("1") & 1, bin(b).count("1") & 1,
            bin(ab).count("1") & 1, bin(a & b).count("1") & 1,
            bin(ab >> za).count("1") & 1 if za != BIG else 0,
            bin(ab >> zb).count("1") & 1 if zb != BIG else 0,
            bin(ab >> zx).count("1") & 1 if zx != BIG else 0,
            (b >> za) & 1 if za != BIG else 0,
            (a >> zb) & 1 if zb != BIG else 0,
            (a >> zx) & 1 if zx != BIG else 0)

def build_map(L):
    h = L.shape[0]; m = {}
    for a in range(h):
        for b in range(h):
            f = feats(a, b); s = int(L[a, b])
            if f in m and m[f] != s: return None      # collision
            m[f] = s
    return m

maps = {}
for n in (8, 9, 10):
    mats, subs = ladder(n)
    plateau = [k for k in subs if len(subs[k]) == 12]
    for k in plateau[1:]:
        maps[(n, k)] = build_map(mats[k])

ref = maps[(9, 3)]
print("TEST F — feature->symbol map identical across levels/dimensions:")
for key, m in maps.items():
    if m is None:
        print(f"  {key}: COLLISION"); continue
    common = set(m) & set(ref)
    agree = all(m[f] == ref[f] for f in common)
    print(f"  {key}: |map|={len(m):4d}  overlap={len(common):4d}  agrees={agree}")

# --- Test G: minimal determining subset (greedy drop) ---
mats, subs = ladder(9)
plateau = [k for k in subs if len(subs[k]) == 12]
L = mats[plateau[1]]; h = L.shape[0]
F = [feats(a, b) for a in range(h) for b in range(h)]
S = [int(L[a, b]) for a in range(h) for b in range(h)]

def determines(idx):
    seen = {}
    for f, s in zip(F, S):
        key = tuple(f[i] for i in idx)
        if key in seen and seen[key] != s: return False
        seen[key] = s
    return True

keep = list(range(len(F[0])))
for i in range(len(F[0])):
    trial = [j for j in keep if j != i]
    if determines(trial): keep = trial
print(f"\nTEST G — minimal features: {len(keep)} of {len(F[0])}")
print("  indices:", keep)
print("  classes:", len({tuple(f[i] for i in keep) for f in F}))

TEST F — feature->symbol map identical across levels/dimensions:
  (8, 3): |map|= 331  overlap= 331  agrees=True
  (8, 4): |map|= 166  overlap= 166  agrees=True
  (8, 5): |map|=  58  overlap=  58  agrees=True
  (9, 3): |map|= 481  overlap= 481  agrees=True
  (9, 4): |map|= 331  overlap= 331  agrees=True
  (9, 5): |map|= 166  overlap= 166  agrees=True
  (9, 6): |map|=  58  overlap=  58  agrees=True
  (10, 3): |map|= 565  overlap= 481  agrees=True
  (10, 4): |map|= 481  overlap= 481  agrees=True
  (10, 5): |map|= 331  overlap= 331  agrees=True
  (10, 6): |map|= 166  overlap= 166  agrees=True
  (10, 7): |map|=  58  overlap=  58  agrees=True

TEST G — minimal features: 8 of 19
  indices: [1, 2, 13, 14, 15, 16, 17, 18]
  classes: 29


In [ ]:
from collections import defaultdict

KEEP = [1, 2, 13, 14, 15, 16, 17, 18]

def feats_min(a, b):
    f = feats(a, b)          # the 19-feature vector
    return tuple(f[i] for i in KEEP)

print("saturation of the minimal map")
seen_global = {}
for n in (8, 9, 10, 11):
    mats, subs = ladder(n)
    plateau = [k for k in subs if len(subs[k]) == 12]
    for k in plateau[1:]:
        L = mats[k]; h = L.shape[0]
        local = {}
        conflict = False
        for a in range(h):
            for b in range(h):
                key = feats_min(a, b); s = int(L[a, b])
                if key in local and local[key] != s: conflict = True
                local[key] = s
                if key in seen_global and seen_global[key] != s: conflict = True
                seen_global[key] = s
        print(f"  n={n} k={k} grid={h:4d}: classes={len(local):3d} "
              f"cumulative={len(seen_global):3d} conflict={conflict}")

print(f"\nTOTAL distinct minimal-feature classes: {len(seen_global)}")
print(f"symbols covered: {sorted(set(seen_global.values()))}")

# extract the two constant tables
print("\n--- L: minimal-feature tuple -> symbol ---")
for key in sorted(seen_global):
    print(f"  {key} -> {seen_global[key]}")

saturation of the minimal map
  n=8 k=3 grid=  32: classes= 29 cumulative= 29 conflict=False
  n=8 k=4 grid=  16: classes= 29 cumulative= 29 conflict=False
  n=8 k=5 grid=   8: classes= 24 cumulative= 29 conflict=False
  n=9 k=3 grid=  64: classes= 29 cumulative= 29 conflict=False
  n=9 k=4 grid=  32: classes= 29 cumulative= 29 conflict=False
  n=9 k=5 grid=  16: classes= 29 cumulative= 29 conflict=False
  n=9 k=6 grid=   8: classes= 24 cumulative= 29 conflict=False
  n=10 k=3 grid= 128: classes= 29 cumulative= 29 conflict=False
  n=10 k=4 grid=  64: classes= 29 cumulative= 29 conflict=False
  n=10 k=5 grid=  32: classes= 29 cumulative= 29 conflict=False
  n=10 k=6 grid=  16: classes= 29 cumulative= 29 conflict=False
  n=10 k=7 grid=   8: classes= 24 cumulative= 29 conflict=False
  n=11 k=3 grid= 256: classes= 29 cumulative= 29 conflict=False
  n=11 k=4 grid= 128: classes= 29 cumulative= 29 conflict=False
  n=11 k=5 grid=  64: classes= 29 cumulative= 29 conflict=False
  n=11 k=6 grid= 

In [ ]:
"""
substitution_o1.py — second O(1) evaluator, from the substitution state.
Test against the ladder (symbol level) and FastStandard (sign level).
"""
import numpy as np

def nu2(x): return (x & -x).bit_length() - 1
def pc(x):  return int(x).bit_count()

# ---------------------------------------------------------------------------
# L : 8 Boolean features -> plateau symbol   (29 rows, saturated n=8..11)
#     key = (b==0, a==b, p_a, p_b, p_x, q_a, q_b, q_x)
# ---------------------------------------------------------------------------
L_TABLE = {
    (0,0,0,0,0,0,0,0): 10, (0,0,0,0,0,0,0,1):  6, (0,0,0,0,0,0,1,1): 12,
    (0,0,0,0,0,1,0,0):  2, (0,0,0,0,0,1,1,0):  8, (0,0,0,0,0,1,1,1):  4,
    (0,0,0,0,1,1,1,0):  4, (0,0,0,0,1,1,1,1):  8, (0,0,0,1,0,0,0,1): 12,
    (0,0,0,1,0,0,1,1):  6, (0,0,0,1,1,0,0,0): 11, (0,0,0,1,1,1,0,0):  3,
    (0,0,1,0,0,0,0,0):  2, (0,0,1,0,0,1,0,0): 10, (0,0,1,0,1,0,0,1):  1,
    (0,0,1,0,1,0,1,1):  7, (0,0,1,1,0,1,1,0):  9, (0,0,1,1,0,1,1,1):  5,
    (0,0,1,1,1,0,0,0):  3, (0,0,1,1,1,0,0,1):  7, (0,0,1,1,1,0,1,1):  1,
    (0,0,1,1,1,1,0,0): 11, (0,0,1,1,1,1,1,0):  5, (0,0,1,1,1,1,1,1):  9,
    (0,1,0,0,0,1,1,0):  4, (0,1,1,1,0,1,1,0):  5,
    (1,0,0,0,0,0,0,1): 12, (1,0,1,0,1,0,0,1):  7,
    (1,1,0,0,0,0,0,0): 12,
}

def plateau_symbol(a, b):
    """O(1) plateau symbol at block position (a,b). None = table miss."""
    a, b = int(a), int(b)
    if a < 0 or b < 0:
        raise ValueError("block coordinates must be >= 0")
    ab, ax = a | b, a ^ b

    def p(x):        # parity of ab above tz(x); 0 when x == 0
        return pc(ab >> nu2(x)) & 1 if x else 0

    def q(x, y):     # bit tz(x) of y; 0 when x == 0
        return (y >> nu2(x)) & 1 if x else 0

    key = (int(b == 0), int(a == b),
           p(a), p(b), p(ax),
           q(a, b), q(b, a), q(ax, a))
    return L_TABLE.get(key)

def sign_from_state(i, j, P):
    """Second O(1) evaluator. P: symbol -> 8x8 sign block."""
    s = plateau_symbol(i >> 3, j >> 3)
    if s is None:
        return None
    return int(P[s][i & 7][j & 7])

# ---------------------------------------------------------------------------
# ladder + extraction of P
# ---------------------------------------------------------------------------
def sgn_ref(i, j):
    """Definition def:o1, unmodified — the previous O(1)."""
    if i == 0 or j == 0: return 1
    if i == j: return -1
    t, ti, tj = nu2(i ^ j), nu2(i), nu2(j)
    if t >= ti and t >= tj: k = t + 1; s = 2*((i >> t) & 1) - 1
    elif ti >= tj:
        k = ti + 1; b = (j >> ti) & 1
        jl = j & ((1 << ti) - 1); d = 1 if jl == 0 else -1
        s = d if b == 0 else -d
    else: k = tj + 1; s = 1 - 2*((i >> tj) & 1)
    if pc((i | j) >> k) & 1: s = -s
    return s

def ladder(n):
    d = 1 << n
    M = np.array([[sgn_ref(i, j) for j in range(d)] for i in range(d)], dtype=np.int64)
    mats, subs = [M], {}
    lvl = 0
    while M.shape[0] >= 2:
        lvl += 1
        h = M.shape[0] // 2
        pats = {}
        for a in range(h):
            for b in range(h):
                pats.setdefault(tuple(int(v) for v in M[2*a:2*a+2, 2*b:2*b+2].ravel()), None)
        keys = sorted(pats)
        rank = {k: r+1 for r, k in enumerate(keys)}
        M = np.array([[rank[tuple(int(v) for v in M[2*a:2*a+2, 2*b:2*b+2].ravel())]
                       for b in range(h)] for a in range(h)], dtype=np.int64)
        subs[lvl] = tuple(keys); mats.append(M)
    return mats, subs

def extract_P(n=9, level=3):
    """symbol -> 8x8 sign block, read off the sign matrix at first occurrence."""
    mats, _ = ladder(n)
    S, Lm = mats[0], mats[level]
    scale = 1 << level
    P = {}
    for a in range(Lm.shape[0]):
        for b in range(Lm.shape[1]):
            s = int(Lm[a, b])
            if s not in P:
                P[s] = S[scale*a:scale*a+scale, scale*b:scale*b+scale].copy()
    return P

In [ ]:
def test_symbol_vs_ladder(dims=(8, 9, 10)):
    """New O(1) symbol vs substitution-derived labels, every plateau cell."""
    total = miss = wrong = 0
    for n in dims:
        mats, subs = ladder(n)
        plateau = [k for k in subs if len(subs[k]) == 12]
        for k in plateau[1:]:
            Lm = mats[k]; h = Lm.shape[0]
            for a in range(h):
                for b in range(h):
                    total += 1
                    got = plateau_symbol(a, b)
                    if got is None: miss += 1
                    elif got != int(Lm[a, b]): wrong += 1
    print(f"symbol level: {total} cells, misses={miss}, wrong={wrong}")
    assert miss == 0 and wrong == 0

def test_sign_vs_fast_standard(P, dims=(4, 6, 8)):
    """New O(1) sign vs previous O(1), exhaustive."""
    total = miss = wrong = 0
    for n in dims:
        d = 1 << n
        for i in range(d):
            for j in range(d):
                total += 1
                got = sign_from_state(i, j, P)
                if got is None: miss += 1
                elif got != sgn_ref(i, j): wrong += 1
    print(f"sign level:   {total} pairs, misses={miss}, wrong={wrong}")
    assert miss == 0 and wrong == 0

def test_structural_identities(P):
    # a=0, b>0 -> symbols 10, 11 : sigma(0,j)=+1, so row 0 all +1
    for s in (10, 11):
        assert all(int(P[s][0][c]) == 1 for c in range(8)), f"identity row, symbol {s}"
    # b=0, a>0 -> symbols 12, 7 : sigma(i,0)=+1, so column 0 all +1
    for s in (12, 7):
        assert all(int(P[s][r][0]) == 1 for r in range(8)), f"identity col, symbol {s}"
    # a=b=0 -> symbol 12, origin is +1
    assert int(P[12][0][0]) == 1
    # a=b>0 -> symbols 4, 5 : diagonal blocks, -1 off origin
    for s in (4, 5):
        assert all(int(P[s][r][r]) == -1 for r in range(1, 8)), f"diag block {s}"
    print("structural identities: OK")
def test_wide_coverage(P, bits=(64, 200, 1000), trials=2000):
    """Wide indices: distinguish table miss from wrong sign."""
    import random
    for B in bits:
        rng = random.Random(B)
        miss = wrong = 0
        for _ in range(trials):
            i, j = rng.getrandbits(B), rng.getrandbits(B)
            got = sign_from_state(i, j, P)
            if got is None: miss += 1
            elif got != sgn_ref(i, j): wrong += 1
        print(f"B={B:5d}: misses={miss}/{trials}, wrong={wrong}")
        assert wrong == 0, "a covered class gave the WRONG sign — formula bug"

if __name__ == "__main__":
    test_symbol_vs_ladder()
    P = extract_P()
    print(f"P extracted: {len(P)} symbols, {P[next(iter(P))].shape} blocks")
    test_sign_vs_fast_standard(P)
    test_structural_identities(P)
    test_wide_coverage(P)

symbol level: 28608 cells, misses=0, wrong=0
P extracted: 12 symbols, (8, 8) blocks
sign level:   69888 pairs, misses=0, wrong=0
structural identities: OK
B=   64: misses=0/2000, wrong=0
B=  200: misses=0/2000, wrong=0
B= 1000: misses=0/2000, wrong=0


In [ ]:
mats, subs = ladder(9)
plateau = [k for k in subs if len(subs[k]) == 12]
sigma = {r+1: pat for r, pat in enumerate(subs[plateau[1]])}

print("self-position check:")
for S, pat in sorted(sigma.items()):
    print(f"  {S:2d} -> {pat}   top-left=={S}: {pat[0] == S}")
print("all self-positioned:", all(pat[0] == S for S, pat in sigma.items()))

# and the scaling identity directly on the label matrices
L = mats[plateau[1]]
h = L.shape[0] // 2
print("L(2a,2b) == L(a,b):",
      all(int(L[2*a, 2*b]) == int(mats[plateau[1]+1][a, b])
            for a in range(h) for b in range(h)))

self-position check:
   1 -> (1, 2, 6, 8)   top-left==1: True
   2 -> (2, 3, 1, 9)   top-left==2: True
   3 -> (3, 2, 12, 4)   top-left==3: True
   4 -> (4, 3, 7, 5)   top-left==4: True
   5 -> (5, 10, 6, 4)   top-left==5: True
   6 -> (6, 11, 1, 5)   top-left==6: True
   7 -> (7, 2, 12, 8)   top-left==7: True
   8 -> (8, 3, 7, 9)   top-left==8: True
   9 -> (9, 10, 6, 8)   top-left==9: True
  10 -> (10, 11, 1, 9)   top-left==10: True
  11 -> (11, 10, 12, 4)   top-left==11: True
  12 -> (12, 11, 7, 5)   top-left==12: True
all self-positioned: True
L(2a,2b) == L(a,b): True
